1. Feature Extracion


In [24]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Load dataset
df = pd.read_csv("Churn_Modelling.csv")

# Drop unwanted columns
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Features and target
X = df.drop("Exited", axis=1)
y = df["Exited"]

# Label Encoding
label_encoder_gender = LabelEncoder()

X["Gender"] = label_encoder_gender.fit_transform(X["Gender"])

# One Hot Encoding
ct = ColumnTransformer(
    transformers=[
        ("geo_encoder", OneHotEncoder(drop="first"), ["Geography"])
    ],
    remainder="passthrough"
)

X = ct.fit_transform(X)

# Get column names
geo_columns = ct.named_transformers_[
    "geo_encoder"
].get_feature_names_out(["Geography"])

remaining_columns = [
    "CreditScore",
    "Gender",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary"
]

all_columns = list(geo_columns) + remaining_columns

# Convert back to DataFrame
X = pd.DataFrame(X, columns=all_columns)

print(X.head())

   Geography_Germany  Geography_Spain  CreditScore  Gender   Age  Tenure  \
0                0.0              0.0        619.0     0.0  42.0     2.0   
1                0.0              1.0        608.0     0.0  41.0     1.0   
2                0.0              0.0        502.0     0.0  42.0     8.0   
3                0.0              0.0        699.0     0.0  39.0     1.0   
4                0.0              1.0        850.0     0.0  43.0     2.0   

     Balance  NumOfProducts  HasCrCard  IsActiveMember  EstimatedSalary  
0       0.00            1.0        1.0             1.0        101348.88  
1   83807.86            1.0        0.0             1.0        112542.58  
2  159660.80            3.0        1.0             0.0        113931.57  
3       0.00            2.0        0.0             0.0         93826.63  
4  125510.82            1.0        1.0             1.0         79084.10  


storing in pkl file 


In [6]:
import pickle

# Save Label Encoder
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)

# Save One Hot Encoder
with open("one_hot_encoder_geo.pkl", "wb") as file:
    pickle.dump(ct, file)

print("Encoders saved successfully!")

Encoders saved successfully!


train and test data divide and store in scalar.pkl file

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Scaling
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Store scaler
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

print("Scaler saved successfully!")

Scaler saved successfully!


Building the ANN model

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [9]:
model = Sequential([
    
    # Hidden Layer 1
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    
    # Hidden Layer 2
    Dense(32, activation='relu'),
    
    # Output Layer
    Dense(1, activation='sigmoid')
])

c:\Users\hp\Desktop\ANN_project\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [12]:
from tensorflow.keras.callbacks import TensorBoard
import datetime
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [13]:
tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping_callback = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [15]:
history = model.fit(
    X_train,
    y_train,

    validation_data=(X_test, y_test),

    epochs=100,
    batch_size=32,

    callbacks=[
        tensorboard_callback,
        early_stopping_callback
    ]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8098 - loss: 0.4530 - val_accuracy: 0.8350 - val_loss: 0.3962
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8384 - loss: 0.3896 - val_accuracy: 0.8490 - val_loss: 0.3643
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8546 - loss: 0.3593 - val_accuracy: 0.8515 - val_loss: 0.3522
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8618 - loss: 0.3467 - val_accuracy: 0.8555 - val_loss: 0.3449
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8597 - loss: 0.3414 - val_accuracy: 0.8640 - val_loss: 0.3431
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8589 - loss: 0.3377 - val_accuracy: 0.8590 - val_loss: 0.3437
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8633 - loss: 0.3345 - val_accuracy: 0.8565 - val_loss: 0.3479
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8631 - loss: 0.3315 - val_accu

In [16]:
model.save("ann_model.h5")

In [25]:
!pip install setuptools

Runnning Tensorboard

In [2]:
%reload_ext tensorboard


In [3]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 10168), started 0:40:19 ago. (Use '!kill 10168' to kill it.)

In [4]:
!kill 10168

kill: 10168: No such process


In [5]:
%reload_ext tensorboard

In [6]:
%tensorboard --logdir logs/fit --port 6007